# Interpretation

In [ ]:
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# We need these from scikit-learn to recreate our test data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Setup complete.")

In [ ]:
# Config
FEATURES_DATA_DIR = Path("data/03_features")
MODELS_DIR = Path("models")

# Loading the entire trained pipeline
print("Loading the trained pipeline...")
pipeline = joblib.load(MODELS_DIR / "full_pipeline.pkl")

# Labels
label_encoder = joblib.load(MODELS_DIR / "label_encoder.pkl")

# Loading data
print("Loading raw features and metadata...")
loaded_data = joblib.load(FEATURES_DATA_DIR / "features_and_metadata.pkl")
text_embeddings = loaded_data['text_embeddings']
metadata = loaded_data['metadata']
y = loaded_data['target']

# Recreate the DataFrame
X_df = pd.concat([
    pd.DataFrame(text_embeddings),
    metadata[['CWE']].reset_index(drop=True)
], axis=1)
X_df.columns = X_df.columns.astype(str)

print("Artifacts loaded successfully.")

In [ ]:
print("Recreating the exact test set used for evaluation...")
y_encoded = label_encoder.fit_transform(y)

# Use the same random_state to get the identical split
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded
)

print(f"Test set recreated with shape: {X_test.shape}")


In [ ]:
print("Creating SHAP Kernel Explainer...")

# SHAP needs a function that takes data and returns model probabilities
def predict_proba_func(data):
    # We need to provide data as a DataFrame with correct column names
    data_df = pd.DataFrame(data, columns=X_train.columns)
    return pipeline.predict_proba(data_df)

# We use a sample of the training data as the background dataset for the explainer
# This helps SHAP understand the baseline values for each feature
background_data = shap.sample(X_train, 100) # Use 100 samples for background

explainer = shap.KernelExplainer(predict_proba_func, background_data)

print("Explainer created.")

In [ ]:
print("Calculating SHAP values for a sample of the test set...")
print("!!! This step can be very slow !!!")

# Let's explain 25 instances from our test set
X_test_sample = X_test.sample(25, random_state=42)

shap_values = explainer.shap_values(X_test_sample)

print("SHAP values calculated.")

In [ ]:
print("--- Global Feature Importance ---")
# Plot the bar chart for overall feature importance
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", class_names=label_encoder.classes_)

In [ ]:
print("--- Global Feature Importance ---")
# Plot the bar chart for overall feature importance
shap.summary_plot(shap_values, X_test_sample, plot_type="bar", class_names=label_encoder.classes_)

In [ ]:
print("\n--- Explaining a Single Prediction (Force Plot) ---")

# Get the SHAP values for the explainer's base value
base_value = explainer.expected_value[critical_class_index]

# Create a force plot for the first sample in our explanation set
shap.force_plot(base_value, shap_values[critical_class_index][0,:], X_test_sample.iloc[0,:], matplotlib=True)